# CONFIGURATION

In [1]:
import pandas as pd
import ast
import numpy as np 
from rapidfuzz import process, fuzz

# EXTRACT (E) STEP
- Scedule Originates from API, downloaded as a CSV to make robust to loss of the API. Data will not change. 
- Transcripts Derived in an earlier workbook from processed recordings. Linkable by the talk name in those cases where scedule item is a recorded talk. 

In [5]:
# Define Extract from CSV function to load data to a Pandas DataFrame
def extract_data_from_csv(csv_file_path): 
    """
    Extracts data from a CSV file and returns a Pandas DataFrame.
    
    Parameters:
        csv_file_path (str): The path to the CSV file.
    """
    try:
        # Read the CSV file into a Pandas DataFrame
        df = pd.read_csv(csv_file_path, encoding="utf-8")
        print(f"Data extracted successfully from '{csv_file_path}'.")
        print(f"Initial DataFrame shape: {df.shape}")
        print("\nInitial Data Head:")
        display(df.head(1))
        return df
    except FileNotFoundError:
        print(f" Error: The file '{csv_file_path}' was not found. Make sure it's in the same directory.")
        df = None
    except Exception as e:
        print(f" Error during CSV read: {e}")
        df = None

    if df is None:
        raise SystemExit("Exiting ETL process due to extraction failure.")

# Robustly load csv to pandas
Schedule_CSV_PATH = "../../EMF_Camp/Raw_data/schedule.csv"
Transcripts_CSV_PATH = "../../EMF_Camp/Raw_data/transcripts.csv"

print("\n--- Step 1: Extraction (E) ---")
schedule_df = extract_data_from_csv(Schedule_CSV_PATH)
transcripts_df = extract_data_from_csv(Transcripts_CSV_PATH)


--- Step 1: Extraction (E) ---
Data extracted successfully from '../../EMF_Camp/Raw_data/schedule.csv'.
Initial DataFrame shape: (550, 19)

Initial Data Head:


,id,type,names,pronouns,title,description,short_description,video_privacy,is_fave,official_content,slug,link,occurrences,cost,equipment,age_range,drop_in,family_friendly,content_note
0,384,performance,"Field-FX organisers, DJ Deathboy",NaN,Demoscene Livecode Jam,We are going to code exciting visual effects i...,Visual livecoding jam with a DJ,public,False,False,demoscene-livecode-jam,https://www.emfcamp.org/schedule/2026/384-demo...,"[{'occurrence_num': 1, 'start_date': '2026-07-...",NaN,NaN,NaN,NaN,NaN,NaN


Data extracted successfully from '../../EMF_Camp/Raw_data/transcripts.csv'.
Initial DataFrame shape: (129, 2)

Initial Data Head:


,title,transcript
0,3D Modelmaking ＂for girls＂ - The overlooked en...,[00:00] Hello.\n[00:17] Hello.\n[00:19] My nam...


# TRANSFORM (T) STEP
- Match titles accounting for minor diffrences. Ensure all Transcripts are paired with their coresponding talk. 

In [8]:
# --- 2. TRANSFORMATION (T) ---
print("\n--- Step 2: Transformation (T) ---")

def closest_transcript_title(title):
    """Fuzzy-matches transcript titles against schedule titles. 
    Returns: Best match, its score, and index in the list of transcript titles.
    Used """
    match = process.extractOne(
        title,
        transcript_titles,
        scorer=fuzz.WRatio,
        score_cutoff=90
    )
    if match:
        return match[0], match[1], match[2]
    return None, 0, None

def closest_schedule_title(title):
    """Find closest schedule title for unmatched transcripts
    Returns: Closest schedule title, its type, and similarity score."""
    match = process.extractOne(
        title,
        schedule_df["title"].dropna().tolist(),
        scorer=fuzz.WRatio
    )
    if match:
        row = schedule_df[schedule_df["title"] == match[0]].iloc[0]
        return match[0], row["type"], match[1]
    return None, None, 0

# Prepare transcript titles for matching
transcript_titles = transcripts_df["title"].dropna().tolist()


# --- DATA PREPERATION ---
# Separate talks and performances from other schedule entries. Testing and context indicates these are the only categories for transcripts
matchable_events = schedule_df[schedule_df["type"].isin(["talk", "performance"])].copy()
other_events = schedule_df[~schedule_df["type"].isin(["talk", "performance"])].copy()
print(f"Talks and performances: {len(matchable_events)}")

# Collect list of transcript titles for later troubleshooting
transcript_titles = transcripts_df["title"].dropna().tolist()


# --- MATCH TITLES ---
# Fuzzy-match schedule titles against transcript titles
matchable_events[["matched_title", "match_score", "match_index"]] = (
    matchable_events["title"].apply(
        lambda x: pd.Series(closest_transcript_title(x))
    )
)

# --- ENRICH DATA ---
# Merge transcripts onto talks and performances
enriched_events = pd.merge(
    matchable_events,
    transcripts_df,
    left_on="matched_title",
    right_on="title",
    how="left",
    suffixes=("_schedule", "_transcript")
)


# --- IDENTIFY UNMATCHED TRANSCRIPTS ---
# Capture matched transcript titles BEFORE cleaning up the merged DataFrame
matched_titles = enriched_events["title_transcript"].dropna().unique()
# Find genuinely unmatched transcripts
unmatched_transcripts = transcripts_df[~transcripts_df["title"].isin(matched_titles)].copy()

# Find closest schedule title and event type for unmatched transcripts
results = unmatched_transcripts["title"].apply(closest_schedule_title).tolist()


# --- MATCHING STATS AND TROUBLESHOOTING ---
results_df = pd.DataFrame(
    results,
    columns=[
        "closest_schedule_title",
        "schedule_type",
        "similarity_score"
    ],
    index=unmatched_transcripts.index
)
unmatched_transcripts[["closest_schedule_title", "schedule_type", "similarity_score"]] = results_df

# Display unmatched transcripts for troubleshooting
print("\nDataframe of Transcripts where a coresponding event in schedule could not be found")
display(
    unmatched_transcripts[
        [
            "title",
            "closest_schedule_title",
            "schedule_type",
            "similarity_score"
        ]
    ].sort_values(
        "similarity_score",
        ascending=False
    )
)
matched_count = enriched_events["transcript"].notna().sum()
unmatched_count = len(unmatched_transcripts)
total_transcripts = len(transcripts_df)

print(f"Total transcripts: {total_transcripts}")
print(f"Matched transcripts: {matched_count}")
print(f"Unmatched transcripts: {unmatched_count}")
print(f"Match rate: {matched_count / total_transcripts * 100:.1f}%")


# --- CLEANUP AND RECOMBINE ---
# Remove temporary matching columns and duplicate transcript title
enriched_events = enriched_events.drop(
    columns=[
        "title_transcript",
        "matched_title",
        "match_score",
        "match_index"
    ],
    errors="ignore"
)
# Rename original schedule title back to title
enriched_events = enriched_events.rename(
    columns={"title_schedule": "title"}
)
# Recombine enriched talks and performances with other schedule entries
enriched_schedule = pd.concat(
    [enriched_events, other_events],
    ignore_index=True,
    sort=False
)

display(enriched_schedule.iloc[1])


--- Step 2: Transformation (T) ---
Talks and performances: 228

Dataframe of Transcripts where a coresponding event in schedule could not be found


,title,closest_schedule_title,schedule_type,similarity_score


Total transcripts: 129
Matched transcripts: 129
Unmatched transcripts: 0
Match rate: 100.0%


id                                                                  82
type                                                              talk
names                                                  James Singleton
pronouns                                                        he/him
title                       💚 Our Family's Electrotech Journey 🌞🏡🔌⛽️🚘️
description          The story of how we went all electric, ditched...
short_description    How we went all electric, ditched fossil fuels...
video_privacy                                                   public
is_fave                                                          False
official_content                                                  True
slug                                   our-familys-electrotech-journey
link                 https://www.emfcamp.org/schedule/2026/82-our-f...
occurrences          [{'occurrence_num': 1, 'start_date': '2026-07-...
cost                                                               NaN
equipm

# Transform (Part 2): Expand Combined Schedule/Talk Table to unique Events.
- (Work in progress)
- occurences column contains JSON which includes details about each instance where the event ran. 
    - In cases where event ran multiple times this coresponds to multiple date times and other data. 
    - The goal of this section is to expand the new enriched Scedule table to have one occurence in time per row with each row having its own start and finish time. 
    - Table should also derive a flag for multiple occurences to help with analysis down the line preventing doublecounting. 

In [22]:
# T2: IDENTIFY MULTIPLE OCCURRENCES AND EXPAND SCHEDULE

# ---------------------------------------------------------
# 1. IMPORT, VALIDATE AND IDENTIFY MULTIPLE OCCURRENCES
# ---------------------------------------------------------
enriched_schedule["occurrences"] = enriched_schedule["occurrences"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
assert enriched_schedule["occurrences"].apply(
    lambda x: isinstance(x, list) and all(isinstance(i, dict) for i in x)
).all(), "Invalid occurrences structure."

enriched_schedule["multiple_occurrences"] = enriched_schedule[
    "occurrences"   
].str.len().gt(1).map({True: "Y", False: "N"})


# ---------------------------------------------------------
# 2. EXPAND SCHEDULE INTO INDIVIDUAL OCCURRENCE ROWS
# ---------------------------------------------------------
expanded_rows = [ # This pattern is nested list comprehension, It is a dense way to get all of the itterable parts in one line. It is equivalent to a nested for loop but more compact.
    {
        **row.drop( ## Drop the columns that will be replaced by occurrence-specific data
            ["occurrences", "start_date", "end_date", "occurrence_data"],
            errors="ignore" # Catches case where these columns don't exist in the DataFrame yet
        ).to_dict(),

        "source_row_index": index,
        "occurrence_data": occurrence.copy() if occurrence else None,
        "occurrence_number": number if occurrence else pd.NA,
        "occurrence_count": len(occurrences),
        "start_date": occurrence.get("start_date") if occurrence else pd.NaT,
        "end_date": occurrence.get("end_date") if occurrence else pd.NaT
    }

    for index, row in enriched_schedule.iterrows()
    for occurrences in [row["occurrences"]]
    for number, occurrence in enumerate(occurrences or [None], start=1)
]

expanded_schedule = pd.DataFrame(expanded_rows)

expanded_schedule[["start_date", "end_date"]] = expanded_schedule[
    ["start_date", "end_date"]
].apply(pd.to_datetime, errors="raise")


# ---------------------------------------------------------
# 3. CREATE MULTIPLE OCCURRENCES TROUBLESHOOTING DATAFRAME
# ---------------------------------------------------------
multiple_occurrences_df = expanded_schedule.query(
    "multiple_occurrences == 'Y'"
).copy()

# ---------------------------------------------------------
# 4. TRANSFORMATION RESULTS AND VALIDATION
# ---------------------------------------------------------
print("\n--- T2 OCCURRENCE EXPANSION RESULTS ---")

print(f"Original rows: {len(enriched_schedule)}")
print(f"Expanded rows: {len(expanded_schedule)}")
print(f"Additional rows: {len(expanded_schedule) - len(enriched_schedule)}")
print(f"Size of multiple occurrences DataFrame: {len(multiple_occurrences_df)}")

print("\n--- DATE COMPLETENESS ---")
for column in ["start_date", "end_date"]:
    print(f"{column}: {expanded_schedule[column].notna().sum()} populated | {expanded_schedule[column].isna().sum()} missing")
assert len(expanded_schedule) == sum(
    max(1, len(x)) for x in enriched_schedule["occurrences"]
)

assert pd.api.types.is_datetime64_any_dtype(expanded_schedule["start_date"])
assert pd.api.types.is_datetime64_any_dtype(expanded_schedule["end_date"])

assert (multiple_occurrences_df["multiple_occurrences"] == "Y").all()

print("\nT2 TRANSFORMATION COMPLETE.")


# ---------------------------------------------------------
# 5. DISPLAY RESULTS AND EXPORT DTYPES
# ---------------------------------------------------------
Abridges_columns = ["id", "type","names","title","short_description","video_privacy","transcript",
    "multiple_occurrences","start_date","end_date","occurrence_data","occurrence_number",
    "occurrence_count"]

print("\n Display head of new expanded schedule.")
display(expanded_schedule[Abridges_columns].head(5))

print("\n Display rows with multiple occurrences for troubleshooting.")
display(multiple_occurrences_df[Abridges_columns].head(5))

print("\n showing dtypes of expanded schedule DataFrame:")
dtype_report = "\n".join(
    f"{i}. {col}: {dtype}"
    for i, (col, dtype) in enumerate(expanded_schedule.dtypes.items(), 1)
)

print("\n Missing date Troublehsooting")
display(
    expanded_schedule[expanded_schedule["start_date"].isna() | expanded_schedule["end_date"].isna()][Abridges_columns]
)

print(dtype_report)


--- T2 OCCURRENCE EXPANSION RESULTS ---
Original rows: 550
Expanded rows: 585
Additional rows: 35
Size of multiple occurrences DataFrame: 51

--- DATE COMPLETENESS ---
start_date: 583 populated | 2 missing
end_date: 583 populated | 2 missing

T2 TRANSFORMATION COMPLETE.

 Display head of new expanded schedule.


,id,type,names,title,short_description,video_privacy,transcript,multiple_occurrences,start_date,end_date,occurrence_data,occurrence_number,occurrence_count
0,384,performance,"Field-FX organisers, DJ Deathboy",Demoscene Livecode Jam,Visual livecoding jam with a DJ,public,NaN,N,2026-07-17 19:30:00,2026-07-17 20:30:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,1
1,82,talk,James Singleton,💚 Our Family's Electrotech Journey 🌞🏡🔌⛽️🚘️,"How we went all electric, ditched fossil fuels...",public,"[00:00] Thank you very much, everyone. Yeah, s...",N,2026-07-18 12:40:00,2026-07-18 13:10:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,1
2,98,talk,Johanna,So - how does the Internet really work,Learn about the fascinating and fun world of c...,public,[00:00] So this is an intro level talk so if y...,N,2026-07-19 12:40:00,2026-07-19 13:10:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,1
3,431,talk,Kiffy,The wonderful world of art toys,Dead Beat Kiffy presents a brief history of th...,public,NaN,N,2026-07-18 18:00:00,2026-07-18 18:30:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,1
4,157,talk,Andy Jenkinson,The Machine That Says No: 1-Bit Music on the Z...,ZX Spectrum 1-bit music from a tiny on/off sig...,public,[00:00] Thank you. Hello. So this talk is a wh...,N,2026-07-18 12:00:00,2026-07-18 12:30:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,1



 Display rows with multiple occurrences for troubleshooting.


,id,type,names,title,short_description,video_privacy,transcript,multiple_occurrences,start_date,end_date,occurrence_data,occurrence_number,occurrence_count
87,376,talk,Thor,Lightning Talks,NaN,public,[00:00] We have a lightning talk.\n[00:21] We ...,Y,2026-07-17 16:50:00,2026-07-17 17:50:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,2
88,376,talk,Thor,Lightning Talks,NaN,public,[00:00] We have a lightning talk.\n[00:21] We ...,Y,2026-07-18 19:50:00,2026-07-18 20:30:00,"{'occurrence_num': 2, 'start_date': '2026-07-1...",2,2
229,165,workshop,Zoë Hutber,Aardman Modelling Workshops (all ages),Build your own clay model of an Aardman charac...,none,NaN,Y,2026-07-18 13:00:00,2026-07-18 14:00:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,2
230,165,workshop,Zoë Hutber,Aardman Modelling Workshops (all ages),Build your own clay model of an Aardman charac...,none,NaN,Y,2026-07-18 15:00:00,2026-07-18 16:00:00,"{'occurrence_num': 2, 'start_date': '2026-07-1...",2,2
269,190,workshop,"Julie, Jon and Emily Spriggs",Board Game Tent,"We bring board games, or you bring board games...",public,NaN,Y,2026-07-18 15:00:00,2026-07-18 17:00:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,3



 showing dtypes of expanded schedule DataFrame:

 Missing date Troublehsooting


,id,type,names,title,short_description,video_privacy,transcript,multiple_occurrences,start_date,end_date,occurrence_data,occurrence_number,occurrence_count
77,175,performance,Karina Townsend,"DIY, modular bagpipes with sound activated las...","Modular, pneumatic sounds with rubber glove ba...",public,NaN,N,NaT,NaT,None,<NA>,0
134,554,talk,Sharon Kinsella,Infrastructure and Light,A talk about the core ideas in my oil painting...,public,NaN,N,NaT,NaT,None,<NA>,0


1. id: int64
2. type: str
3. names: str
4. pronouns: str
5. title: str
6. description: str
7. short_description: str
8. video_privacy: str
9. is_fave: bool
10. official_content: bool
11. slug: str
12. link: str
13. cost: str
14. equipment: str
15. age_range: str
16. drop_in: object
17. family_friendly: object
18. content_note: str
19. transcript: str
20. multiple_occurrences: str
21. source_row_index: int64
22. occurrence_data: object
23. occurrence_number: object
24. occurrence_count: int64
25. start_date: datetime64[us]
26. end_date: datetime64[us]


missing date is as expected as original also does not have it. 

In [25]:
# T3: Additional Derived Columns
expanded_schedule["Duration_min"] = expanded_schedule.apply(
    lambda row: (row["end_date"] - row["start_date"]).total_seconds() / 60
    if pd.notna(row["start_date"]) and pd.notna(row["end_date"])
    else pd.NA,
    axis=1
)
display(expanded_schedule[["start_date", "end_date", "Duration_min"]].head(5))

# Confirm column datatypes
print("\n showing dtypes of expanded schedule DataFrame:")
dtype_report = "\n".join(
    f"{i}. {col}: {dtype}"
    for i, (col, dtype) in enumerate(expanded_schedule.dtypes.items(), 1)
)
print(dtype_report)

,start_date,end_date,Duration_min
0,2026-07-17 19:30:00,2026-07-17 20:30:00,60.0
1,2026-07-18 12:40:00,2026-07-18 13:10:00,30.0
2,2026-07-19 12:40:00,2026-07-19 13:10:00,30.0
3,2026-07-18 18:00:00,2026-07-18 18:30:00,30.0
4,2026-07-18 12:00:00,2026-07-18 12:30:00,30.0



 showing dtypes of expanded schedule DataFrame:
1. id: int64
2. type: str
3. names: str
4. pronouns: str
5. title: str
6. description: str
7. short_description: str
8. video_privacy: str
9. is_fave: bool
10. official_content: bool
11. slug: str
12. link: str
13. cost: str
14. equipment: str
15. age_range: str
16. drop_in: object
17. family_friendly: object
18. content_note: str
19. transcript: str
20. multiple_occurrences: str
21. source_row_index: int64
22. occurrence_data: object
23. occurrence_number: object
24. occurrence_count: int64
25. start_date: datetime64[us]
26. end_date: datetime64[us]
27. Duration_min: object


In [ ]:
# Filtering and Data Quality Checks/tweaks
expanded_schedule["cost"] = expanded_schedule["cost"].fillna("Not Avalible")
display(expanded_schedule["cost"].unique()) ## Needs some entity resolution, multiple ways of saying free. 

<StringArray>
[                                                                                                                  'Not Avalible',
                                                                                                                            '£15',
                                                                                                                            'nil',
                                                                                                                           'Free',
                                                                                   'free, donations to St John Ambulance welcome',
                                                                                                                           'free',
                                                                                                                              '-',
                                                                     

# Todo 
> Still need to massage data types and perform cleaning to match the template for Postgress

# LOAD (L)

## Get Enviroment variables 

In [2]:
from dotenv import load_dotenv, find_dotenv
import os
import psycopg2
from psycopg2 import sql
import psycopg2.extras as extras

load_dotenv(find_dotenv())

CSV_FILE = os.getenv('CSV_FILE')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
TARGET_TABLE = "etl_transformed_schedule"

## Postgress DB CONNECTION

In [3]:
#Establish Connection to PostgreSQL Database
try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    print(f"Successfully connected to PostgreSQL database: {DB_NAME}")

except psycopg2.Error as e:
    print(f"Database connection failed. Error: {e}")
    print("Please check your database service and credentials.")

Successfully connected to PostgreSQL database: EMF_Camp


##  CREATE TABLE BLUEPRINT

```
1. id: int64 x
2. type: str x
3. names: str x
4. pronouns: str x
5. title: str x
6. description: str x
7. short_description: str x
8. video_privacy: str x
9. is_fave: bool x
10. official_content: bool x
11. slug: str /
12. link: str x
13. cost: str x
14. equipment: str x
15. age_range: str x
16. drop_in: object x
17. family_friendly: object x
18. content_note: str x
19. transcript: str x
20. multiple_occurrences: str /
21. source_row_index: int64 / 
22. occurrence_data: object  x 
23. occurrence_number: object x
24. occurrence_count: int64 x
25. start_date: datetime64[us] x
26. end_date: datetime64[us] x
27. Duration_min: object x 
```

In [ ]:
# Create the target table structure in PostgreSQL if it doesn't exist
create_table_query = sql.SQL(f"""
        CREATE TABLE IF NOT EXISTS {TARGET_TABLE}(
            id INTEGER UNIQUE,
            type VARCHAR(255),
            title VARCHAR(255),
            short_description VARCHAR(255),
            description VARCHAR(255),
            names VARCHAR(255),
            pronouns VARCHAR(100),
            link VARCHAR(255),  -- Confirm if URL has a specific type in PostgreSQL, otherwise use VARCHAR
            transcript VARCHAR(255),
            start_date TIMESTAMP WITH TIME ZONE,
            end_date TIMESTAMP WITH TIME ZONE,
            Duration_min FLOAT,
            drop_in BOOLEAN,
            occurrence_count INTEGER,
            occurrence_number INTEGER,
            content_note VARCHAR(255),
            family_friendly BOOLEAN,
            age_range VARCHAR(100),
            cost VARCHAR(100),
            equipment VARCHAR(255),
            official_content BOOLEAN,
            video_privacy VARCHAR(100),
            is_fave BOOLEAN,
            occurrence_data JSONB, -- Should confirm if this is the correct type. 
            load_timestamp TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
        );
    """
)

try:
    with conn.cursor() as cur:
        cur.execute(create_table_query)
        conn.commit()
    print(f"Table '{TARGET_TABLE}' already exists or has been successully created!")
except psycopg2.Error as e:
    print(f" Error creating table: {e}")
    conn.rollback()

Table 'etl_transformed_schedule' already exists or has been successully created!


## Insert data into table

In [ ]:
# Convert the DataFrame to a list of tuples for psycopg2's execute_batch
data_tuples = [tuple(row) for row in df_load.values]
# Define the INSERT query structure
cols = ', '.join(df_load.columns)
placeholders = ', '.join(['%s'] * len(df_load.columns))
# Define columns to update (all except the unique key 'order_id')
update_cols = [col for col in df_load.columns if col not in ['order_id']]
set_clause = ', '.join([f"{col} = EXCLUDED.{col}" for col in update_cols])
insert_query = sql.SQL(
    f"""
    INSERT INTO {TARGET_TABLE} ({cols}) 
    VALUES ({placeholders}) 
    ON CONFLICT (order_id) 
    DO UPDATE SET {set_clause}
    """
)
try:
    with conn.cursor() as cur:
        # Use execute_batch for efficient bulk insertion
        extras.execute_batch(cur, insert_query, data_tuples)
        conn.commit()
        print(f"Successfully attempted to load {len(data_tuples)} records into '{TARGET_TABLE}'. Conflicting records were updated.")

except psycopg2.Error as e:
    print(f"Error during data loading: {e}")
    conn.rollback()

finally:
    if conn:
        conn.close()
        print(" Database connection closed.")

print("\n--- ETL Process Complete ---")